# 03 — Llama-3.2-3B-Instruct: baseline + LoRA fine-tuning

Pipeline (all under the 8B-parameter limit, **3.2B params**):

1. **Baseline**: score the base model on `val` and `test`, 0-shot and few-shot.
2. **Fine-tune**: QLoRA (4-bit base + LoRA adapters) on `train`. After every epoch it measures `val` accuracy and keeps the best adapter.
3. **Final eval**: best fine-tuned adapter on `val` and `test`, compared side by side with the baseline.

**How answers are scored:** instead of generating text, the model's next-token probabilities for `1`, `2`, `3`, `4` are compared and the highest wins. So every question gets a valid answer, with no parsing.

**Hardware:** needs an NVIDIA GPU. A free **Colab T4** or **Kaggle T4/P100** is enough. Expect roughly 30–45 min end to end on a T4.

**If the session gets cut off** (e.g. a Colab usage limit): results, checkpoints and the adapter are saved to Google Drive (Colab) or `/kaggle/working` (Kaggle). Run the notebook again and finished evaluations are skipped, and training continues from the last completed epoch.

**Before running:** accept the licence at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct and create a token at https://huggingface.co/settings/tokens. Then add it as a secret named `HF_TOKEN` (Colab: key icon in the left sidebar; Kaggle: Add-ons → Secrets).

**Private GitHub repo:** also add a secret named `GITHUB_TOKEN`. Create it at https://github.com/settings/personal-access-tokens: choose "Fine-grained token", select only this repository, and grant *Contents: Read-only*.

## 1. Environment

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # one GPU is plenty for 3B (Kaggle gives two)

GH_USER = "Imalsha-Ranasinghe"
REPO    = f"github.com/{GH_USER}/SinhalaMMLU-Shared-Task.git"
BRANCH  = "main"          # the branch to run; it must contain the latest src/ and notebooks

# Kaggle images also contain the google.colab package, so check Kaggle first, by its own env var.
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB  = not IN_KAGGLE and "COLAB_RELEASE_TAG" in os.environ
print("Running on:", "Kaggle" if IN_KAGGLE else "Colab" if IN_COLAB else "local / other")

def get_secret(name):
    # Read a secret from env vars, then Kaggle or Colab secrets
    if os.getenv(name):
        return os.getenv(name).strip()
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name).strip()
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name).strip()
    except Exception as e:
        print(f"[{name}] could not read secret: {type(e).__name__}: {e}")
    return None

def is_project(d):
    # Recognise this repo by its own files. A bare "src" folder is not enough: Kaggle's saved-version
    # runs keep the notebook's source in /kaggle/src, which would be mistaken for the project.
    return (d / "src" / "hf_model.py").exists() and (d / "data").is_dir()

clone_dir = Path.cwd() / "SinhalaMMLU-Shared-Task"
if not (IN_KAGGLE or IN_COLAB) and is_project(Path.cwd()):
    ROOT = Path.cwd()                      # running locally from the repo root
elif not (IN_KAGGLE or IN_COLAB) and is_project(Path.cwd().parent):
    ROOT = Path.cwd().parent               # running locally from notebooks/
else:
    # Colab / Kaggle: always clone fresh so the code matches GitHub. Results live outside this folder.
    if clone_dir.exists():
        shutil.rmtree(clone_dir)
    gh_token = get_secret("GITHUB_TOKEN")
    print("GITHUB_TOKEN found:", bool(gh_token))
    auth = f"{GH_USER}:{gh_token}@" if gh_token else ""
    r = subprocess.run(["git", "clone", "-q", "--branch", BRANCH, f"https://{auth}{REPO}", str(clone_dir)],
                       capture_output=True, text=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if r.returncode != 0:
        err = r.stderr.replace(gh_token, "***") if gh_token else r.stderr
        raise RuntimeError("git clone failed: " + err)
    # don't leave the token in .git/config
    subprocess.run(["git", "-C", str(clone_dir), "remote", "set-url", "origin", f"https://{REPO}"])
    ROOT = clone_dir
    commit = subprocess.run(["git", "-C", str(clone_dir), "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout
    print(f"Cloned {BRANCH} @ {commit.strip()}")

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

# Kaggle/Colab already ship most of these; install only what is missing or too old (fewer ways to fail).
import importlib, time
from packaging.version import Version
REQUIRED = {"transformers": "4.56", "peft": "0.13", "accelerate": "1.0", "bitsandbytes": "0.45"}
missing = []
for pkg, min_version in REQUIRED.items():
    try:
        if Version(importlib.import_module(pkg).__version__) < Version(min_version):
            missing.append(f"{pkg}>={min_version}")
    except Exception:
        missing.append(f"{pkg}>={min_version}")
if missing:
    print("installing:", missing)
    for attempt in (1, 2):                 # a second try covers temporary download errors
        pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], capture_output=True, text=True)
        if pip.returncode == 0:
            break
        time.sleep(10)
    if pip.returncode != 0:                # show pip's own error instead of a bare CalledProcessError
        print(pip.stdout[-3000:], pip.stderr[-3000:], sep="\n")
        raise RuntimeError("pip install failed - see pip's messages above")
print("packages ready:", {p: importlib.import_module(p).__version__ for p in REQUIRED})

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. On Colab: Runtime -> Change runtime type -> T4 GPU"
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")
print("bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
# Hugging Face login (Llama 3.2 is a gated model)
from huggingface_hub import login

token = get_secret("HF_TOKEN")
if token:
    login(token=token)
    print("Logged in to Hugging Face")
else:
    raise RuntimeError("HF_TOKEN not found. Kaggle: Add-ons -> Secrets, add HF_TOKEN and tick its checkbox. "
                       "Colab: key icon -> add HF_TOKEN with Notebook access on.")

## 2. Configuration — edit this cell

In [ ]:
MODEL_ID     = "meta-llama/Llama-3.2-3B-Instruct"   # ungated copy with identical weights: "unsloth/Llama-3.2-3B-Instruct"
SHORT_NAME   = "llama3.2-3b-instruct"
MAX_PARAMS   = 8e9            # project limit
LOAD_IN_4BIT = True
FORCE_FP32   = False          # set True if training shows loss = nan on a T4/P100
SEED         = 42

EVAL_SPLITS    = ["val", "test"]
BASELINE_SHOTS = [0, 3]       # few-shot settings for the base model (examples come from train, same subject); [0] saves GPU time
SCORER_TOKENS  = 8000         # tokens per forward pass at eval time; lower it if you hit CUDA OOM
SANITY_CHECK   = True         # also score 300 English MMLU questions, to tell pipeline bugs apart from weak Sinhala

# --- LoRA fine-tuning ---
DO_TRAIN     = True
AUG_SHUFFLES = 0              # extra copies of each training question with the options reordered (0 = off)
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
# Attention + MLP projections of the language model (the regex skips any image encoder)
LORA_TARGETS = r"^(?!.*vision).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$"
EPOCHS       = 3
LR           = 2e-4
BATCH_SIZE   = 4
GRAD_ACCUM   = 4              # effective batch = 16
MAX_LEN      = 1280           # longest 0-shot training example is ~1250 tokens

# Persistent storage, so a cut-off session can resume instead of starting over.
if IN_KAGGLE:
    PERSIST_DIR = Path("/kaggle/working")          # saved to the version's Output tab
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_DIR = Path("/content/drive/MyDrive/SinhalaMMLU-runs")
else:
    PERSIST_DIR = ROOT

RESULTS_DIR = PERSIST_DIR / "results"
ADAPTER_DIR = PERSIST_DIR / "outputs" / f"{SHORT_NAME}-lora"
CKPT_DIR    = PERSIST_DIR / "outputs" / f"{SHORT_NAME}-checkpoints"
print("Saving results to ", RESULTS_DIR)
print("Saving adapter to ", ADAPTER_DIR)

## 3. Parameter budget check

In [ ]:
from accelerate import init_empty_weights
from transformers import AutoConfig, AutoModelForCausalLM

with init_empty_weights():
    _m = AutoModelForCausalLM.from_config(AutoConfig.from_pretrained(MODEL_ID))
n_params = sum(p.numel() for p in _m.parameters())   # tied embeddings counted once
del _m
print(f"{MODEL_ID}: {n_params / 1e9:.2f}B parameters (limit {MAX_PARAMS / 1e9:.0f}B)")
assert n_params < MAX_PARAMS

## 4. Load data and model

In [ ]:
from src.data import load_jsonl

data = {s: load_jsonl(ROOT / "data" / "splits" / f"{s}.jsonl") for s in ["train", "val", "test"]}
for s, d in data.items():
    print(f"{s:5s}: {len(d)}")

In [ ]:
from src.hf_model import load_model_and_tokenizer, load_with_dtype_check, ChoiceScorer, render_prompt
from src.prompting import build_messages

probe = [build_messages(r) for r in data["val"].head(8).to_dict(orient="records")]
if FORCE_FP32:
    model, tok = load_model_and_tokenizer(MODEL_ID, load_in_4bit=LOAD_IN_4BIT, dtype=torch.float32)
    DTYPE = torch.float32
else:
    model, tok, DTYPE = load_with_dtype_check(MODEL_ID, probe, load_in_4bit=LOAD_IN_4BIT)
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

print(render_prompt(tok, build_messages(data["test"].iloc[0].to_dict())))

## 5. Baseline (no fine-tuning)

In [ ]:
import json
import pandas as pd
from src.prompting import ShotSampler
from src.evaluate import run_eval, compute_metrics, save_metrics

def evaluate(scorer, run_name, shots, splits=EVAL_SPLITS):
    sampler = ShotSampler(data["train"], k=shots, seed=SEED)
    run_dir = RESULTS_DIR / run_name
    rows = {}
    for split_name in splits:
        preds = run_eval(data[split_name], scorer, sampler, run_dir / f"{split_name}_predictions.jsonl", batch_size=64)
        m = compute_metrics(preds)
        save_metrics(m, run_dir / f"{split_name}_metrics.json")
        rows[split_name] = m["accuracy"]
        print(f"  {run_name} / {split_name}: accuracy = {m['accuracy']:.2%}")
    return rows

scorer = ChoiceScorer(model, tok, max_tokens_per_batch=SCORER_TOKENS)
summary = {}
for k in BASELINE_SHOTS:
    summary[f"base {k}-shot"] = evaluate(scorer, f"{SHORT_NAME}_base_{k}shot", k)
pd.DataFrame(summary).T.style.format("{:.2%}")

### Sanity check: is the pipeline correct?

Same model, same prompt, same scoring code, but on **English** MMLU questions. Meta reports ~63% on MMLU for Llama-3.2-3B-Instruct.
If this lands roughly in the 50–65% range, the code is fine and a low Sinhala score reflects the model's Sinhala ability.

In [ ]:
if SANITY_CHECK:
    from huggingface_hub import hf_hub_download

    mmlu = pd.read_parquet(hf_hub_download("cais/mmlu", "all/test-00000-of-00001.parquet", repo_type="dataset"))
    mmlu = mmlu.sample(n=300, random_state=SEED).reset_index(drop=True)
    mmlu = pd.DataFrame({
        "id": [f"mmlu::{i}" for i in range(len(mmlu))],
        "subject": mmlu["subject"], "category": "english_mmlu",
        "question": mmlu["question"], "choices": mmlu["choices"].apply(list),
        "answer": mmlu["answer"] + 1,            # MMLU is 0-based, ours is 1-based
    })
    preds = run_eval(mmlu, scorer, ShotSampler(data["train"], k=0),
                     RESULTS_DIR / f"{SHORT_NAME}_base_english_mmlu" / "predictions.jsonl", batch_size=64)
    acc = preds["correct"].mean()
    print(f"English MMLU, 300 questions, 0-shot: {acc:.1%}")
    print("-> Pipeline OK. The low Sinhala score is the model, not the code." if acc > 0.45
          else "-> Suspiciously low for English: the pipeline may be broken. Share this output.")

## 6. LoRA fine-tuning

In [ ]:
import math
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments
from src.hf_model import build_train_examples, Collator, make_trainer_class, make_val_accuracy_callback
from src.prompting import shuffle_augment

done_file = ADAPTER_DIR / "training_complete.json"

if DO_TRAIN and done_file.exists():
    print(f"Training already finished in an earlier session; loading the best adapter from {ADAPTER_DIR}")
    model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
elif DO_TRAIN:
    torch.manual_seed(SEED)
    if LOAD_IN_4BIT:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.config.use_cache = False
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS, task_type="CAUSAL_LM",
    ))
    model.print_trainable_parameters()

    train_df = shuffle_augment(data["train"], AUG_SHUFFLES, seed=SEED)
    train_examples = build_train_examples(tok, train_df, max_len=MAX_LEN)
    print(f"{len(train_examples)} training examples ({len(data['train'])} questions, {AUG_SHUFFLES} shuffled copies)")
    steps = math.ceil(len(train_examples) / (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS

    args = TrainingArguments(
        output_dir=str(CKPT_DIR),
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_steps=max(1, int(0.05 * steps)),
        logging_steps=10,
        save_strategy="epoch",                   # checkpoint every epoch so a cut-off run can resume
        save_total_limit=1,
        bf16=DTYPE == torch.bfloat16, fp16=DTYPE == torch.float16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        remove_unused_columns=False,
        report_to="none",
        seed=SEED,
    )
    ft_scorer = ChoiceScorer(model, tok, max_tokens_per_batch=SCORER_TOKENS)
    val_cb = make_val_accuracy_callback(ft_scorer, data["val"], ADAPTER_DIR)
    trainer = make_trainer_class()(
        model=model, args=args, train_dataset=train_examples,
        data_collator=Collator(tok.pad_token_id), callbacks=[val_cb],
    )
    resume = any(CKPT_DIR.glob("checkpoint-*"))
    if resume:
        print("Resuming from the last saved epoch in", CKPT_DIR)
    trainer.train(resume_from_checkpoint=True if resume else None)
    tok.save_pretrained(ADAPTER_DIR)

    # The model in memory is from the last epoch; switch to the best epoch's adapter.
    model.load_adapter(str(ADAPTER_DIR), adapter_name="best")
    model.set_adapter("best")
    done_file.write_text(json.dumps(val_cb.history))
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(val_cb.history).to_csv(RESULTS_DIR / f"{SHORT_NAME}_lora_val_history.csv", index=False)
    print(pd.DataFrame(val_cb.history))

## 7. Evaluate the best fine-tuned adapter

In [ ]:
if DO_TRAIN:
    model.config.use_cache = True
    ft_scorer = ChoiceScorer(model, tok, max_tokens_per_batch=SCORER_TOKENS)
    summary["LoRA 0-shot"] = evaluate(ft_scorer, f"{SHORT_NAME}_lora_0shot", 0)

final = pd.DataFrame(summary).T
final.to_csv(RESULTS_DIR / f"{SHORT_NAME}_summary.csv")
final.style.format("{:.2%}")

In [ ]:
# Per-subject test accuracy: baseline vs fine-tuned
def by_subject(run_name, split="test"):
    with open(RESULTS_DIR / run_name / f"{split}_metrics.json", encoding="utf-8") as f:
        return pd.DataFrame(json.load(f)["by_subject"]).set_index("subject")["accuracy"]

runs = {f"base {k}-shot": f"{SHORT_NAME}_base_{k}shot" for k in BASELINE_SHOTS}
if DO_TRAIN:
    runs["LoRA 0-shot"] = f"{SHORT_NAME}_lora_0shot"
pd.DataFrame({name: by_subject(r) for name, r in runs.items()}).sort_values(list(runs)[-1], ascending=False).style.format("{:.2%}")

### Which answers does each model pick?

If a model gives the same number for almost every question, it isn't reading the questions. It has collapsed to the most common answer, which is a typical failure after fine-tuning a model that can't understand the input.

In [ ]:
gold = data["test"]["answer"].value_counts(normalize=True).reindex([1, 2, 3, 4], fill_value=0)
print(f"Random guessing: 25.0%    Always answering {gold.idxmax()}: {gold.max():.1%}  <- the real floor to beat")
dist = {"gold answers": gold}
for name, r in runs.items():
    p = pd.read_json(RESULTS_DIR / r / "test_predictions.jsonl", lines=True)
    dist[name] = p["pred"].value_counts(normalize=True).reindex([1, 2, 3, 4], fill_value=0)
pd.DataFrame(dist).style.format("{:.1%}")

## 8. Getting the files

Everything is already in `PERSIST_DIR` (printed in the config cell):
- **Colab:** Google Drive → `MyDrive/SinhalaMMLU-runs/` (`results/`, `outputs/llama3.2-3b-instruct-lora/`).
- **Kaggle:** the notebook's **Output** tab after a "Save & Run All" version.

To load the adapter again later:

```python
from peft import PeftModel
model, tok = load_model_and_tokenizer(MODEL_ID, load_in_4bit=True)
model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
scorer = ChoiceScorer(model, tok)
```